In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, make_scorer
import pickle


Load trained models and rest data

In [ ]:
with open('models/solar_final.pkl', 'rb') as f:
    solar_model = pickle.load(f)
with open('models/wind_final.pkl', 'rb') as f:
    wind_model = pickle.load(f)
with open('models/load_final.pkl', 'rb') as f:
    load_model = pickle.load(f)


target_train = pd.read_parquet('data/target_train.parquet')
weather_train = pd.read_parquet('data/weather_train.parquet')
network_train = pd.read_parquet('data/network_train.parquet')

In [ ]:




def create_price_features(y_load, y_solar, y_wind, weather_df, network_df):
    """Create features for price model (uses other predictions!)"""
    df = pd.DataFrame(index=weather_df.index)
    
    # Weather signals
    df['temp_mean'] = weather_df['temp_mean']
    df['ssrd_mean'] = weather_df['ssrd_mean']
    df['wind_mean'] = weather_df['wind_mean']
    
    # Supply/demand signals (from other model predictions)
    df['load'] = y_load
    df['solar'] = y_solar
    df['wind'] = y_wind
    df['renewable_pct'] = (y_solar + y_wind) / y_load  # Supply/demand ratio
    
    # Fuel prices (direct cost signal)
    df['eex_carbon'] = network_df['EEX_CARBON']
    df['eex_coal'] = network_df['EEX_COAL']
    df['eex_gas'] = network_df['EEX_GAS_PEG']
    
    # Availability (what's available to produce)
    df['avail_coal'] = network_df['FR_availability_coal']
    df['avail_gas'] = network_df['FR_availability_gas']
    df['avail_hydro'] = network_df['FR_availability_hydro']
    df['avail_nuclear'] = network_df['FR_availability_nuclear']
    
    # Time features
    dt = pd.to_datetime(weather_df.index)
    hour = dt.hour
    dow = dt.dayofweek
    month = dt.month
    
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    df['month_sin'] = np.sin(2 * np.pi * month / 12)
    df['month_cos'] = np.cos(2 * np.pi * month / 12)
    
    return df




In [ ]:

# ==================== 3. PREPARE FEATURES ====================
print("\n[Step 3] Preparing features for price model...")

weather_processed = prepare_weather(weather_train)
# Get validation data (2024)
X_valid_solar = ... # from solar notebook
X_valid_wind = ... # from wind notebook
X_valid_load = ... # from load notebook

# Generate predictions on 2024 using trained models
y_pred_solar = solar_model.predict(X_valid_solar)
y_pred_wind = wind_model.predict(X_valid_wind)
y_pred_load = load_model.predict(X_valid_load)

# Create price features
price_features = create_price_features(
    y_load=y_pred_load,
    y_solar=y_pred_solar,
    y_wind=y_pred_wind,
    weather_df=weather_processed.loc['2024'],
    network_df=network_train.loc['2024']
)

price_df = target_train[['FR_price_actual']].loc['2024'].join(price_features)

# ==================== 4. SPLIT DATA ====================
print("\n[Step 4] Train/validation split...")

X_train_price = price_df.loc['2020':'2023'].drop('FR_price_actual', axis=1)
y_train_price = price_df.loc['2020':'2023']['FR_price_actual']

X_valid_price = price_df.loc['2024'].drop('FR_price_actual', axis=1)
y_valid_price = price_df.loc['2024']['FR_price_actual']

# ==================== 5. TRAIN MODEL ====================
print("\n[Step 5] Training price model...")

model_price = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=32,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)

model_price.fit(X_train_price, y_train_price)

# ==================== 6. EVALUATE ====================
print("\n[Step 6] Evaluation...")

y_pred_price = model_price.predict(X_valid_price)

rmse = np.sqrt(mean_squared_error(y_valid_price, y_pred_price))
mae = mean_absolute_error(y_valid_price, y_pred_price)
wmape = np.sum(np.abs(y_valid_price - y_pred_price)) / np.sum(np.abs(y_valid_price))
mape = mean_absolute_percentage_error(y_valid_price, y_pred_price)
r2 = r2_score(y_valid_price, y_pred_price)

print(f"\nPrice Model Validation:")
print(f"  RMSE: {rmse:.2f} €/MWh")
print(f"  MAE: {mae:.2f} €/MWh")
print(f"  WMAPE: {wmape:.4f}")
print(f"  MAPE: {mape:.4f}")
print(f"  R²: {r2:.4f}")

# ==================== 7. CROSS-VALIDATION ====================
print("\n[Step 7] Cross-validation...")

tscv = TimeSeriesSplit(n_splits=3)
rmse_scores = -cross_val_score(
    model_price, X_cv, y_cv, cv=tscv,
    scoring=make_scorer(
        lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
        greater_is_better=False
    )
)

print(f"  CV RMSE: {rmse_scores.mean():.2f} €/MWh")

# ==================== 8. SAVE ====================
print("\n[Step 8] Saving price model...")

with open('models/price_final.pkl', 'wb') as f:
    pickle.dump(model_price, f)

print("✅ Price model saved!")